# Bandingkan budget epoch pada dataset_retrain

Melatih satu kali sampai budget terpanjang, lalu menyimpan checkpoint val_loss terbaik untuk tiap budget. Karena data order dan bobot awal sama, selisih hasil murni berasal dari jumlah epoch, bukan dari run yang berbeda.

Tiap checkpoint dievaluasi pada validation dan test. Pemenang dipilih dari val_loss, jadi angka test tidak ikut memilih model.

Jalankan setelah `dataset_retrain/` siap dari `train_banana.ipynb`. Hasil ditulis ke `artifacts_retrain_cmp/`; `artifacts_retrain/` dan aset Flutter tidak disentuh.

Riwayat epoch ditulis ke `history.csv` selama training, jadi bila kernel berhenti di tengah jalan, angka epoch yang sudah lewat tetap tersimpan. Setelah checkpoint ada, isi `EVALUATE_ONLY = True` untuk mengevaluasi ulang tanpa training.

Estimasi di mesin tanpa GPU: sekitar 32 detik per epoch, jadi budget 100 epoch sekitar 53 menit. Evaluasi memuat checkpoint satu per satu dan melepasnya setelah dipakai, karena menahan dua model sekaligus di mesin 16 GB berisiko membuat kernel berhenti.


## Import dan konfigurasi

`EPOCH_BUDGETS` berisi budget yang dibandingkan. `PATIENCE` mengatur EarlyStopping pada val_loss; isi 0 untuk mematikannya. `LIMIT` membatasi gambar per kelas per split untuk uji cepat; isi 0 untuk memakai semua. `EVALUATE_ONLY` melewati training dan hanya mengevaluasi checkpoint yang sudah ada di `OUT_DIR`.


In [1]:
import gc
import json
import math
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "dataset_retrain").is_dir():
    BASE_DIR = BASE_DIR / "training"
if not (BASE_DIR / "dataset_retrain").is_dir():
    raise FileNotFoundError(
        "Jalankan notebook dari root proyek atau folder training yang berisi dataset_retrain.")

CLASSES = ["cordana", "healthy", "pestalotiopsis", "sigatoka"]
VALID_EXTENSIONS = {".jpeg", ".jpg", ".png"}
SPLITS = ("train", "val", "test")

DATASET_DIR = BASE_DIR / "dataset_retrain"
OUT_DIR = BASE_DIR / "artifacts_retrain_cmp"
EPOCH_BUDGETS = [20, 100]
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
PATIENCE = 10         # EarlyStopping pada val_loss; 0 mematikannya
LIMIT = 0             # batasi gambar per kelas per split untuk uji cepat; 0 memakai semua
EVALUATE_ONLY = False # True: lewati training, pakai checkpoint di OUT_DIR
print("TensorFlow:", tf.__version__)


## `BestWithinBudget`

Checkpoint hanya diperbarui selama epoch masih di dalam budget, sehingga folder `e20/` menyimpan model terbaik pada epoch 1 sampai 20 dan `e100/` menyimpan model terbaik pada epoch 1 sampai 100.


In [1]:
class BestWithinBudget(tf.keras.callbacks.Callback):
    """Simpan model hanya selama budget epoch masih terbuka."""

    def __init__(self, filepath, epoch_limit, monitor="val_loss"):
        super().__init__()
        self.filepath = Path(filepath)
        self.epoch_limit = epoch_limit
        self.monitor = monitor
        self.best = math.inf
        self.best_epoch = None

    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1
        if current_epoch > self.epoch_limit:
            return
        value = (logs or {}).get(self.monitor)
        if value is not None and value < self.best:
            self.best = value
            self.best_epoch = current_epoch
            self.model.save(self.filepath)


## `load_rgb`

Resize nearest memakai koordinat `floor(indeks * ukuran_asli / 224)` agar sama dengan Dart di Flutter. Normalisasi ke [-1, 1] ada di dalam model.


In [1]:
def load_rgb(path):
    with Image.open(path) as source:
        pixels = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))
    ys = np.arange(IMG_SIZE) * pixels.shape[0] // IMG_SIZE
    xs = np.arange(IMG_SIZE) * pixels.shape[1] // IMG_SIZE
    return pixels[ys[:, None], xs[None, :]].astype(np.float32)


## `collect_paths`


In [1]:
def collect_paths(dataset_dir, split, limit):
    paths, targets = [], []
    for index, name in enumerate(CLASSES):
        files = sorted(p for p in (dataset_dir / split / name).glob("*")
                       if p.suffix.lower() in VALID_EXTENSIONS)
        if limit:
            files = files[:limit]
        if not files:
            raise ValueError(f"Dataset kosong: {split}/{name} di {dataset_dir}")
        paths.extend(files)
        targets.extend([index] * len(files))
    return paths, targets


## `make_dataset`


In [1]:
def make_dataset(paths, targets, shuffle, seed, batch_size):
    def samples():
        for path, target in zip(paths, targets):
            yield load_rgb(path), np.int32(target)

    ds = tf.data.Dataset.from_generator(samples, output_signature=(
        tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
        tf.TensorSpec((), tf.int32)))
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(paths)))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=seed)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


## `build_model`

Backbone ImageNet dibekukan. Augmentasi hanya aktif saat training. Label mengikuti urutan `CLASSES`.


In [1]:
def build_model(seed):
    tf.keras.utils.set_random_seed(seed)
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
    ], name="augmentation")
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="rgb")
    x = augmentation(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax",
                                    name="probabilities")(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


## `evaluate`

Confusion matrix: baris kelas sebenarnya, kolom prediksi, sesuai urutan `CLASSES`.


In [1]:
def evaluate(model, ds, targets):
    metrics = model.evaluate(ds, return_dict=True, verbose=0)
    probabilities = model.predict(ds, verbose=0)
    predictions = probabilities.argmax(axis=1)
    confusion = tf.math.confusion_matrix(
        targets, predictions, num_classes=len(CLASSES)).numpy()
    return ({k: float(v) for k, v in metrics.items()}, confusion.tolist())


## `release_model`

Menutup model dan membersihkan sesi Keras, supaya checkpoint berikutnya dimuat tanpa menumpuk memori.


In [1]:
def release_model(model):
    del model
    gc.collect()
    tf.keras.backend.clear_session()


## `print_confusion_matrix`

Confusion matrix test: baris = kelas sebenarnya, kolom = prediksi, sesuai urutan `CLASSES`. Metrik per kelas dihitung langsung dari matriks, jadi tidak perlu scikit-learn.


In [1]:
def print_confusion_matrix(confusion, labels, title):
    matrix = np.asarray(confusion)
    width = max(len(name) for name in labels) + 2
    print(title)
    print(f"{'':>{width}s}"
          + "".join(f"{name:>{width}s}" for name in labels)
          + f"{'total':>{width}s}")
    for index, name in enumerate(labels):
        cells = "".join(f"{int(v):>{width}d}" for v in matrix[index])
        print(f"{name:>{width}s}" + cells + f"{int(matrix[index].sum()):>{width}d}")

    print()
    print(f"{'kelas':>{width}s} {'precision':>9s} {'recall':>9s} {'f1':>9s} {'support':>8s}")
    for index, name in enumerate(labels):
        support = int(matrix[index].sum())
        predicted = int(matrix[:, index].sum())
        true_positive = int(matrix[index, index])
        precision = true_positive / predicted if predicted else 0.0
        recall = true_positive / support if support else 0.0
        denominator = precision + recall
        f1 = 2 * precision * recall / denominator if denominator else 0.0
        print(f"{name:>{width}s} {precision:9.3f} {recall:9.3f} {f1:9.3f} {support:8d}")

    total = int(matrix.sum())
    accuracy = float(np.trace(matrix)) / total if total else 0.0
    print(f"{'akurasi':>{width}s} {'':>9s} {'':>9s} {accuracy:9.3f} {total:8d}")


## `write_history_csv` dan `read_history_csv`

`write_history_csv` menulis riwayat per epoch. `read_history_csv` membacanya kembali, dipakai mode `EVALUATE_ONLY` dan untuk menghitung epoch terbaik tiap budget tanpa training.


In [1]:
def write_history_csv(path, history):
    keys = ["epoch", "loss", "accuracy", "val_loss", "val_accuracy"]
    lines = [",".join(keys)]
    for epoch in range(len(history["loss"])):
        row = [str(epoch + 1)] + [repr(float(history[k][epoch])) for k in keys[1:]]
        lines.append(",".join(row))
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def read_history_csv(path):
    keys = ["loss", "accuracy", "val_loss", "val_accuracy"]
    history = {key: [] for key in keys}
    for line in path.read_text(encoding="utf-8").splitlines()[1:]:
        if not line.strip():
            continue
        parts = line.split(",")
        for key, value in zip(keys, parts[1:]):
            history[key].append(float(value))
    return history


def best_epoch_within(history, budget):
    """Epoch dengan val_loss terkecil pada rentang 1..budget."""
    window = history["val_loss"][:budget]
    if not window:
        return None
    return int(np.argmin(window)) + 1


## `write_comparison_csv`


In [1]:
def write_comparison_csv(path, results):
    keys = ["budget", "best_epoch", "val_loss", "val_accuracy",
            "test_loss", "test_accuracy"]
    lines = [",".join(keys)]
    for r in results:
        lines.append(",".join([
            str(r["budget"]), str(r["best_epoch"]),
            f"{r['val_loss']:.6f}", f"{r['val_accuracy']:.6f}",
            f"{r['test_loss']:.6f}", f"{r['test_accuracy']:.6f}",
        ]))
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


## `write_curves_svg`

Menulis plot loss dan accuracy train/val sebagai SVG, jadi tidak butuh matplotlib. Garis putus-putus menandai akhir budget terpendek.


In [1]:
def write_curves_svg(path, history, marker_epoch):
    epochs = list(range(1, len(history["loss"]) + 1))
    panels = [
        ("Loss", [("loss", "train", "#1f77b4"), ("val_loss", "val", "#d62728")]),
        ("Accuracy", [("accuracy", "train", "#1f77b4"),
                      ("val_accuracy", "val", "#d62728")]),
    ]
    width, height = 780, 560
    left, right, top, gap = 70, 560, 40, 70
    panel_h = (height - top - 40 - gap) / 2
    e_min, e_max = min(epochs), max(epochs)

    def x_of(epoch):
        if e_max == e_min:
            return (left + right) / 2
        return left + (epoch - e_min) / (e_max - e_min) * (right - left)

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" '
        f'height="{height}" viewBox="0 0 {width} {height}">',
        f'<rect width="{width}" height="{height}" fill="#ffffff"/>',
        f'<g font-family="sans-serif" font-size="12" fill="#222222">',
    ]
    for index, (title, series) in enumerate(panels):
        y0 = top + index * (panel_h + gap)
        y1 = y0 + panel_h
        values = [history[key][e - 1] for key in (s[0] for s in series)
                  for e in epochs]
        lo, hi = min(values), max(values)
        pad = (hi - lo) * 0.08 or 0.01
        lo, hi = lo - pad, hi + pad

        def y_of(value, lo=lo, hi=hi, y0=y0, y1=y1):
            return y1 - (value - lo) / (hi - lo) * (y1 - y0)

        parts.append(f'<text x="{left}" y="{y0 - 10}" font-size="14">{title}</text>')
        for fraction in (0.0, 0.5, 1.0):
            value = lo + (hi - lo) * fraction
            y = y_of(value)
            parts.append(f'<line x1="{left}" y1="{y:.1f}" x2="{right}" '
                         f'y2="{y:.1f}" stroke="#e6e6e6"/>')
            parts.append(f'<text x="{left - 8}" y="{y + 4:.1f}" '
                         f'text-anchor="end" fill="#666666">{value:.3f}</text>')
        parts.append(f'<line x1="{left}" y1="{y1}" x2="{right}" y2="{y1}" '
                     f'stroke="#999999"/>')
        parts.append(f'<line x1="{left}" y1="{y0}" x2="{left}" y2="{y1}" '
                     f'stroke="#999999"/>')

        for key, label, color in series:
            points = " ".join(f"{x_of(e):.1f},{y_of(history[key][e - 1]):.1f}"
                              for e in epochs)
            parts.append(f'<polyline points="{points}" fill="none" '
                         f'stroke="{color}" stroke-width="2"/>')
            legend_y = y0 + 14 + series.index((key, label, color)) * 18
            parts.append(f'<line x1="{right + 12}" y1="{legend_y}" '
                         f'x2="{right + 34}" y2="{legend_y}" stroke="{color}" '
                         f'stroke-width="2"/>')
            parts.append(f'<text x="{right + 40}" y="{legend_y + 4}" '
                         f'fill="#333333">{label}</text>')

        if marker_epoch is not None:
            mx = x_of(marker_epoch)
            parts.append(f'<line x1="{mx:.1f}" y1="{y0}" x2="{mx:.1f}" y2="{y1}" '
                         f'stroke="#888888" stroke-dasharray="4 3"/>')
            parts.append(f'<text x="{mx + 4:.1f}" y="{y0 + 12}" '
                         f'fill="#666666">epoch {marker_epoch}</text>')

    parts.append(f'<text x="{left}" y="{height - 12}" fill="#666666">'
                 f'epoch (1..{e_max})</text>')
    parts.append("</g></svg>")
    path.write_text("\n".join(parts) + "\n", encoding="utf-8")


## Alur perbandingan

`main()` merangkai semua langkah: muat dataset, latih sekali sampai budget terpanjang (kecuali `EVALUATE_ONLY`), evaluasi tiap checkpoint pada val dan test, lalu tulis laporan. Konfigurasi diambil dari sel pertama.

Model training dilepas sebelum tahap evaluasi, dan tiap checkpoint dievaluasi lalu dilepas, supaya hanya satu model yang aktif di memori pada satu waktu.


In [1]:
def main():
    budgets = sorted({int(b) for b in EPOCH_BUDGETS})
    if not budgets or budgets[0] < 1:
        raise ValueError("Budget epoch harus bilangan bulat positif.")
    max_epochs = budgets[-1]
    if not DATASET_DIR.is_dir():
        raise FileNotFoundError(f"Dataset tidak ditemukan: {DATASET_DIR}")

    print("=" * 64)
    print("  PERBANDINGAN BUDGET EPOCH")
    print("=" * 64)
    print(f"  Dataset    : {DATASET_DIR}")
    print(f"  Output     : {OUT_DIR}")
    print(f"  Budget     : {budgets}")
    print(f"  Seed       : {SEED}   Patience: {PATIENCE}")
    print(f"  Mode       : {'evaluasi saja' if EVALUATE_ONLY else 'training + evaluasi'}")

    print("\n[1/4] Memuat dataset ...")
    split_paths, split_targets = {}, {}
    for split in SPLITS:
        paths, targets = collect_paths(DATASET_DIR, split, LIMIT)
        split_paths[split], split_targets[split] = paths, targets
        print(f"  {split:5s}: {len(paths)} gambar")

    train_ds = make_dataset(split_paths["train"], split_targets["train"], True,
                            SEED, BATCH_SIZE)
    val_ds = make_dataset(split_paths["val"], split_targets["val"], False,
                          SEED, BATCH_SIZE)
    test_ds = make_dataset(split_paths["test"], split_targets["test"], False,
                           SEED, BATCH_SIZE)

    history_path = OUT_DIR / "history.csv"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    if EVALUATE_ONLY:
        print("\n[2/4] Melewati training, membaca riwayat dari disk ...")
        if not history_path.is_file():
            raise FileNotFoundError(
                f"EVALUATE_ONLY butuh {history_path}. Jalankan tanpa mode ini lebih dulu.")
        history_data = read_history_csv(history_path)
        epochs_run = len(history_data["loss"])
        stopped_early = epochs_run < max_epochs
        snapshots = {b: type("S", (), {"best_epoch": best_epoch_within(history_data, b)})()
                     for b in budgets}
    else:
        print("\n[2/4] Membangun model dan training ...")
        model = build_model(SEED)
        snapshots = {}
        callbacks = []
        for budget in budgets:
            target = OUT_DIR / f"e{budget}" / "banana_best.keras"
            target.parent.mkdir(parents=True, exist_ok=True)
            snapshot = BestWithinBudget(target, budget)
            snapshots[budget] = snapshot
            callbacks.append(snapshot)
        if PATIENCE > 0:
            callbacks.append(tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=PATIENCE, restore_best_weights=False))
        callbacks.append(tf.keras.callbacks.CSVLogger(str(history_path)))

        history = model.fit(train_ds, validation_data=val_ds, epochs=max_epochs,
                            shuffle=False, verbose=1, callbacks=callbacks)
        history_data = dict(history.history)
        epochs_run = len(history_data["loss"])
        stopped_early = epochs_run < max_epochs
        release_model(model)

    print("\n[3/4] Mengevaluasi checkpoint tiap budget ...")
    results = []
    for budget in budgets:
        snapshot = snapshots[budget]
        checkpoint = OUT_DIR / f"e{budget}" / "banana_best.keras"
        if not checkpoint.is_file():
            print(f"  e{budget}: tidak ada checkpoint, dilewati.")
            continue
        restored = tf.keras.models.load_model(checkpoint)
        expected_epoch = best_epoch_within(history_data, budget)
        if expected_epoch != snapshot.best_epoch:
            print(f"  [!] e{budget}: checkpoint dari epoch {snapshot.best_epoch}, "
                  f"padahal val_loss terbaik ada di epoch {expected_epoch}.")
        val_metrics, _ = evaluate(restored, val_ds, split_targets["val"])
        test_metrics, confusion = evaluate(restored, test_ds, split_targets["test"])
        evaluation = {"budget": budget, "best_epoch": snapshot.best_epoch,
                      "checkpoint": str(checkpoint.relative_to(OUT_DIR)),
                      "labels": [c.capitalize() for c in CLASSES],
                      "validation": val_metrics, "test": test_metrics,
                      "confusion_matrix": confusion}
        (checkpoint.parent / "evaluation.json").write_text(
            json.dumps(evaluation, indent=2), encoding="utf-8")
        results.append({
            "budget": budget,
            "best_epoch": snapshot.best_epoch,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "confusion_matrix": confusion,
            "checkpoint": str(checkpoint.relative_to(OUT_DIR)),
        })
        print(f"  e{budget}: epoch terbaik {snapshot.best_epoch}, "
              f"val_loss {val_metrics['loss']:.4f}")
        release_model(restored)

    if not results:
        raise RuntimeError("Tidak ada checkpoint yang bisa dievaluasi.")

    print("\n[4/4] Menulis laporan ...")
    write_history_csv(history_path, history_data)
    write_comparison_csv(OUT_DIR / "comparison.csv", results)
    write_curves_svg(OUT_DIR / "curves.svg", history_data, budgets[0])

    selected = min(results, key=lambda r: r["val_loss"])
    comparison = {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "patience": PATIENCE,
        "epochs_requested": max_epochs,
        "epochs_run": epochs_run,
        "stopped_early": stopped_early,
        "budgets": budgets,
        "selection_metric": "val_loss",
        "selected_budget": selected["budget"],
        "results": results,
    }
    (OUT_DIR / "comparison.json").write_text(
        json.dumps(comparison, indent=2), encoding="utf-8")

    print("\n" + "=" * 64)
    print("  HASIL")
    print("=" * 64)
    print(f"  Epoch dijalankan : {epochs_run} dari {max_epochs}"
          + ("  (berhenti awal)" if stopped_early else ""))

    labels = [c.capitalize() for c in CLASSES]
    for r in results:
        print()
        print_confusion_matrix(r["confusion_matrix"], labels,
                               f"Confusion matrix test (budget {r['budget']} epoch, "
                               f"epoch terbaik {r['best_epoch']})")

    print(f"\n  {'Budget':>6s} {'BestEp':>6s} {'ValLoss':>9s} {'ValAcc':>8s} "
          f"{'TestLoss':>9s} {'TestAcc':>8s}")
    print("  " + "-" * 50)
    for r in results:
        print(f"  {r['budget']:6d} {r['best_epoch']:6d} {r['val_loss']:9.4f} "
              f"{r['val_accuracy']:8.4f} {r['test_loss']:9.4f} "
              f"{r['test_accuracy']:8.4f}")
    print("  " + "-" * 50)
    print(f"  Dipilih lewat val_loss: {selected['budget']} epoch "
          f"(val_loss {selected['val_loss']:.4f})")
    print("\n  File:")
    print(f"    history      : {OUT_DIR / 'history.csv'}")
    print(f"    perbandingan : {OUT_DIR / 'comparison.csv'}")
    print(f"    kurva        : {OUT_DIR / 'curves.svg'}")
    print(f"    detail       : {OUT_DIR / 'comparison.json'}")


## Jalankan perbandingan

Ubah `EPOCH_BUDGETS` di sel konfigurasi bila perlu, lalu jalankan sel ini. Untuk mencoba cepat, set `LIMIT` kecil dan `EPOCH_BUDGETS = [1, 2]` lebih dulu.

Bila kernel berhenti di tengah training, isi `EVALUATE_ONLY = True` di sel konfigurasi dan jalankan ulang sel ini: checkpoint dan `history.csv` yang sudah ada dipakai untuk menulis laporan tanpa mengulang training.


In [1]:
main()


TensorFlow: 2.21.0
  PERBANDINGAN BUDGET EPOCH
  Dataset    : C:\PBL-KEL4_TI3C_SEM5\training\dataset_retrain
  Output     : C:\PBL-KEL4_TI3C_SEM5\training\artifacts_retrain_cmp
  Budget     : [20, 100]
  Seed       : 42   Patience: 10
  Mode       : training + evaluasi

[1/4] Memuat dataset ...
  train: 779 gambar
  val  : 275 gambar
  test : 267 gambar

[2/4] Membangun model dan training ...
Epoch 1/100

 1/49 ━━━━━━━━━━━━━━━━━━━━ 17:11 21s/step - accuracy: 0.3125 - loss: 1.7740
 1/49 ==================== 17:11 21s/step - accuracy: 0.3125 - loss: 1.7740
 2/49 ━━━━━━━━━━━━━━━━━━━━ 25s 542ms/step - accuracy: 0.3125 - loss: 1.7715
 2/49 ==================== 25s 542ms/step - accuracy: 0.3125 - loss: 1.7715
 3/49 ━━━━━━━━━━━━━━━━━━━━ 22s 497ms/step - accuracy: 0.2917 - loss: 1.8719
 3/49 ==================== 22s 497ms/step - accuracy: 0.2917 - loss: 1.8719
 4/49 ━━━━━━━━━━━━━━━━━━━━ 22s 497ms/step - accuracy: 0.3125 - loss: 1.8509
 4/49 ==================== 22s 497ms/step - accuracy: 0.312

C:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
C:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\JpegImagePlugin.py:874: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(

